# Day 05 · Tool 기반 MCP 기능 구현
FastMCP로 행동하는 도구(Tool)를 제대로 만듭니다 — 부작용, 입력 검증과 에러, 구조화된 출력과 annotations, Context와 진행 보고, 그리고 검색을 도구로 노출하기.
Day03~04처럼 별도 프로세스 없이 in-process Client로 바로 검증합니다.

### ctx로 할 수 있는 것 (참조)
| 분류 | ctx 사용 | 방향 |
|---|---|---|
| 로깅 | `ctx.info/debug/warning/error(msg)` | 서버→클라 (통지) |
| 진행 보고 | `ctx.report_progress(p, total, msg)` | 서버→클라 (통지) |
| Sampling | `ctx.sample(messages, ...)` | 서버⇄클라 (왕복) · 클라 LLM |
| Elicitation | `ctx.elicit(msg, response_type)` | 서버⇄클라 (왕복) · 사용자 |
| 자원 접근 | `ctx.read_resource / list_resources / get_prompt` | 서버 내부 |
| 세션 상태 | `ctx.get_state / set_state / delete_state` | 서버 내부 |
| 요청 정보 | `ctx.request_id / session_id / client_id / transport` | 읽기(속성) |


## Lab 0 · 셋업

In [ ]:
%pip install -q fastmcp

In [ ]:
import fastmcp
print("fastmcp", fastmcp.__version__)

from fastmcp import FastMCP, Client, Context
mcp = FastMCP("Day05ToolServer")

## Lab 1 · 부작용 도구
Resource(읽기 전용, Day04)와 달리 Tool은 상태를 바꾸는 행동을 한다. 여기서는 할 일 목록에 저장하고 완료 처리한다.

In [ ]:
TASKS = []

@mcp.tool
def add_task(title: str, priority: str = "보통") -> dict:
    """할 일을 추가한다 (부작용: 목록에 저장)"""
    task = {"id": len(TASKS) + 1, "title": title, "priority": priority, "done": False}
    TASKS.append(task)
    return {"저장됨": task, "총개수": len(TASKS)}

@mcp.tool
def complete_task(task_id: int) -> str:
    """할 일을 완료 처리한다"""
    for t in TASKS:
        if t["id"] == task_id:
            t["done"] = True
            return f"완료: {t['title']}"
    return f"id {task_id} 인 할 일이 없습니다."

async with Client(mcp) as c:
    print((await c.call_tool("add_task", {"title": "슬라이드 검토", "priority": "높음"})).data)
    print((await c.call_tool("add_task", {"title": "노트북 검증"})).data)
    print("완료 처리:", (await c.call_tool("complete_task", {"task_id": 1})).data)

## Lab 2 · 입력 검증과 친절한 에러
잘못된 인자에는 무엇이 왜 틀렸는지 알려주는 에러를 던진다. 클라이언트는 그 메시지를 받는다.

In [ ]:
ALLOWED = {"낮음", "보통", "높음"}

@mcp.tool
def set_priority(task_id: int, priority: str) -> str:
    """우선순위를 바꾼다. 허용값이 아니면 친절한 에러를 던진다."""
    if priority not in ALLOWED:
        raise ValueError(f"priority는 {sorted(ALLOWED)} 중 하나여야 합니다. 받은 값: '{priority}'")
    for t in TASKS:
        if t["id"] == task_id:
            t["priority"] = priority
            return f"{t['title']} 우선순위 → {priority}"
    raise ValueError(f"id {task_id} 인 할 일이 없습니다.")

async with Client(mcp) as c:
    print((await c.call_tool("set_priority", {"task_id": 2, "priority": "낮음"})).data)
    try:
        await c.call_tool("set_priority", {"task_id": 2, "priority": "긴급"})
    except Exception as e:
        print("에러 전달됨:", str(e).splitlines()[-1])

## Lab 3 · 구조화된 출력과 annotations
dict 대신 스키마가 분명한 구조(Pydantic)로 반환하고, 이 도구가 읽기 전용인지 등을 annotations로 알린다.

In [ ]:
from pydantic import BaseModel

class TaskStats(BaseModel):
    total: int
    done: int
    pending: int

@mcp.tool(annotations={"readOnlyHint": True})
def task_stats() -> TaskStats:
    """할 일 통계를 구조화해 반환한다 (읽기 전용)"""
    done = sum(t["done"] for t in TASKS)
    return TaskStats(total=len(TASKS), done=done, pending=len(TASKS) - done)

async with Client(mcp) as c:
    anns = {t.name: t.annotations for t in await c.list_tools()}
    print("task_stats 읽기전용?:", anns["task_stats"].readOnlyHint)
    r = await c.call_tool("task_stats", {})
    print("구조화 출력:", r.data, "| total =", r.data.total)

## Lab 4 · Context와 진행 보고
오래 걸리는 작업은 Context(ctx)로 로그와 진행률을 보고한다. 인자에 `ctx: Context`를 넣으면 자동 주입된다.
클라이언트는 `progress_handler`로 진행 상황을 받는다.

In [ ]:
import asyncio

@mcp.tool
async def bulk_import(count: int, ctx: Context) -> dict:
    """여러 건을 가져오며 진행률을 보고한다"""
    await ctx.info(f"{count}건 가져오기 시작")
    for i in range(count):
        TASKS.append({"id": len(TASKS) + 1, "title": f"가져온 항목 {i+1}", "priority": "보통", "done": False})
        await ctx.report_progress(i + 1, count)
        await asyncio.sleep(0.3)        # 실제 작업(파일/DB/네트워크)을 흉내 — 이게 있어야 진행률이 실시간으로 보인다
    await ctx.info("완료")
    return {"가져옴": count, "총개수": len(TASKS)}

async def on_progress(progress, total, message=None):
    print(f"  진행 {int(progress)}/{int(total)} ({int(progress/total*100)}%)", flush=True)

async with Client(mcp, progress_handler=on_progress) as c:
    print((await c.call_tool("bulk_import", {"count": 5})).data)

## Lab 5 · 검색을 도구로 — 진짜 RAG (청킹·임베딩·검색)
Day02 코퍼스를 가져와 **청킹 → 임베딩 → 벡터 검색**을 `search_docs` **Tool**로, 청크 원문은 `chunk://` **Resource**로 노출합니다.
- 임베딩: CPU에서도 가벼운 `multilingual-e5-small`(118M) — Day02의 그 RAG를 이제 MCP로 감싼다.
- 이 `search_docs`를 **Lab 7에서 LLM이 스스로 호출**하고, **Lab 8에서 sampling으로 답**합니다.
> 설치에 잠깐 걸립니다(sentence-transformers). 모델은 처음 1회만 내려받습니다.


In [ ]:
%pip install -q sentence-transformers langchain-text-splitters

**1) Day02 코퍼스를 가져와 청킹**
Day02에서 만든 사내 문서를 URL로 가져와 200자(겹침 20)로 잘게 나눕니다.

In [ ]:
import urllib.request
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS_URL = "https://raw.githubusercontent.com/TunaLee/nvidia-dli/main/day02/data/corpus.txt"
raw = urllib.request.urlopen(CORPUS_URL, timeout=20).read().decode("utf-8")
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
CHUNKS = []
for block in raw.split("===DOC==="):
    block = block.strip()
    if not block:
        continue
    head, *rest = block.splitlines()
    meta = dict(kv.split(": ", 1) for kv in head.split(" | ") if ": " in kv)
    for piece in splitter.split_text("\n".join(rest).strip()):
        CHUNKS.append({"doc_id": meta.get("doc_id"), "title": meta.get("title"), "text": piece})

print(f"청크 {len(CHUNKS)}개")
print("예:", CHUNKS[0]["doc_id"], CHUNKS[0]["title"], "|", CHUNKS[0]["text"][:40])

**2) 임베딩 — multilingual-e5-small (CPU에서도 가볍다)**
e5는 문서엔 `passage:`, 질문엔 `query:` 접두어를 붙입니다. 처음 1회 모델 다운로드가 있습니다.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("intfloat/multilingual-e5-small")
def embed(texts, kind):
    return embedder.encode([f"{kind}: {t}" for t in texts], normalize_embeddings=True)

EMB = embed([c["text"] for c in CHUNKS], "passage")
print("임베딩 shape:", EMB.shape)

**3) 검색 = Tool (벡터 코사인 유사도)**
질문을 임베딩해 가장 가까운 청크를 찾습니다. 검색은 '연산(행동)'이라 **Tool**.

In [ ]:
def retrieve(query, k=3):
    scores = EMB @ embed([query], "query")[0]
    top = np.argsort(-scores)[:k]
    return [{"id": int(i), "doc_id": CHUNKS[i]["doc_id"], "title": CHUNKS[i]["title"],
             "text": CHUNKS[i]["text"]} for i in top]

@mcp.tool(annotations={"readOnlyHint": True})
def search_docs(query: str, k: int = 3) -> list:
    """질문과 의미가 가까운 사내 문서 청크 상위 k개 (임베딩 코사인 유사도)"""
    return retrieve(query, k)

async with Client(mcp) as c:
    print("검색: '연차 신청은 며칠 전?'")
    for h in (await c.call_tool("search_docs", {"query": "연차 신청은 며칠 전?", "k": 2})).data:
        print(f"  {h['doc_id']} {h['title']} :: {h['text'][:45]}")

**4) 청크 원문 = Resource**
검색으로 찾은 청크의 원문은 id로 읽는 **읽기 전용 리소스**로 노출합니다.

In [ ]:
@mcp.resource("chunk://{cid}", description="청크 원문을 id로 읽는다")
def chunk(cid: str) -> str:
    return CHUNKS[int(cid)]["text"]

async with Client(mcp) as c:
    print((await c.read_resource("chunk://0"))[0].text[:60])

### (옵션) 임베딩도 NVIDIA API로
로컬 e5 대신 NVIDIA 임베딩 모델을 쓰면 RAG 전체가 NVIDIA API로 돕니다. (NVIDIA 클라우드 `integrate.api.nvidia.com` + 아래 토큰 셀의 `client` 필요)
```python
def embed_nvidia(texts, input_type):   # input_type: "query" | "passage"
    r = client.embeddings.create(
        model="nvidia/nv-embedqa-e5-v5", input=texts,
        extra_body={"input_type": input_type, "truncate": "END"})
    return np.array([d.embedding for d in r.data])
# EMB = embed_nvidia([c["text"] for c in CHUNKS], "passage")   # e5 대신 이걸로
```
> 주의: 임베딩 모델이 엔드포인트에 있어야 합니다. 강의용 DGX 프록시가 Qwen(LLM)만이면 로컬 e5를 쓰세요.


## Lab 6 · 도전 (선택)
- **A** 이 서버를 `server.py`로 저장하고 `mcp.run()`을 붙여 터미널에서 `fastmcp dev`로 브라우저 검증.
- **B** dry-run(미리보기) 모드 — 삭제 전 '무엇이 지워질지'를 먼저 보여준다 (Lab 9 확장)
- **C** `search_docs`를 Day02의 `advanced_rag`(하이브리드+리랭커)로 교체해 진짜 검색 도구로.
- **D** Claude Desktop 등 실제 MCP 클라이언트에 이 서버를 연결.

```python
# server.py 로 저장해 실행하는 형태
if __name__ == "__main__":
    mcp.run()   # 기본 stdio
```

## Lab 7 · 실무 — LLM이 MCP 도구를 스스로 호출한다 (NVIDIA Qwen)
지금까지는 **우리가 코드로** `call_tool`을 불렀습니다. 실무의 핵심은 **LLM이 스스로 판단해 도구를 호출**하는 것입니다.
NVIDIA Qwen에게 이 서버의 도구 목록을 function schema로 건네주면, Qwen이 필요한 도구를 골라 부르고 그 결과로 답합니다.

흐름: ① MCP 도구 → OpenAI function schema ② LLM이 도구 선택·호출 ③ 우리가 MCP로 실행 → 결과 반환 ④ LLM이 근거로 최종 답변.
이게 **Agentic RAG의 씨앗** — 3주차에서 이 루프를 LangGraph로 확장합니다.

확인: 검색이 필요한 질문에서 Qwen이 `search_docs`를 스스로 호출하는가?
- **자기수정**: 도구가 친절한 에러를 던지면 그 에러가 LLM에 되돌아가고, LLM이 인자를 고쳐 **다시 부른다**(아래 '긴급'→'높음' 데모).


In [ ]:
# NVIDIA API 토큰 (Day01~02와 동일). Lab 7에서 LLM이 도구를 호출하는 데 씁니다.
# 로컬에서 server.py 실행 시엔 셸 환경변수로 토큰을 넣습니다:
#   PowerShell:  $env:NVIDIA_API_KEY="nvapi-..."   ·   cmd:  set NVIDIA_API_KEY=nvapi-...
import os, getpass, json
from openai import OpenAI
LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or os.getenv("QWEN_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in llm.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl","embed","rerank"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("모델:", LLM_MODEL)

**① MCP 도구 → OpenAI function 스펙으로 변환**
LLM이 이해하는 형식으로 도구 목록을 바꿉니다.

In [ ]:
def to_openai_tools(mcp_tools):
    # MCP 도구(name·description·inputSchema)를 OpenAI function 스펙으로 변환
    return [{"type": "function", "function": {
        "name": t.name, "description": t.description or "", "parameters": t.inputSchema}} for t in mcp_tools]

**② 에이전트 루프**
LLM이 도구를 고르면 실행하고 결과를 다시 넘겨 반복. 도구가 에러를 던지면 그 에러를 되돌려 **자기수정**하게 합니다.

In [ ]:
async def run_agent(question, max_steps=4):
    async with Client(mcp) as client:
        oai_tools = to_openai_tools(await client.list_tools())
        messages = [{"role": "system", "content": "너는 사내 도우미다. 필요하면 제공된 도구로 근거를 찾은 뒤 한국어로 간결히 답하라."},
                    {"role": "user", "content": question}]
        for _ in range(max_steps):
            r = llm.chat.completions.create(model=LLM_MODEL, messages=messages, tools=oai_tools, temperature=0.2, max_tokens=400)
            m = r.choices[0].message
            if not m.tool_calls:                       # 도구를 더 안 부르면 최종 답변
                return m.content
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:                     # LLM이 고른 도구를 MCP로 실제 실행
                args = json.loads(tc.function.arguments)
                print(f"  [LLM → 도구] {tc.function.name}({args})")
                try:
                    res = await client.call_tool(tc.function.name, args)
                    content = json.dumps(res.data, ensure_ascii=False, default=str)
                except Exception as e:                  # 도구가 에러를 던지면 → LLM에 되돌려 '스스로 고치게'
                    content = f"오류: {str(e).splitlines()[-1]}"
                    print(f"       ↳ 오류를 LLM에 전달: {content}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": content})
        return "(반복 한도 초과)"

**③ 실행 — 사내 문서 검색**

In [ ]:
print("Q: 연차는 며칠 전에 신청해야 하나요?")
print("A:", await run_agent("연차는 며칠 전에 신청해야 하나요?"))

**④ 자기수정 데모 — 잘못된 인자를 스스로 고친다**
'긴급'은 허용값이 아니라 에러 → LLM이 '높음'으로 고쳐 재호출(로그가 두 번).

In [ ]:
# 자기수정 데모: '긴급'은 허용값(낮음·보통·높음)이 아니라 set_priority 가 친절한 에러를 던진다.
# → LLM이 그 에러를 읽고 '높음'으로 고쳐 다시 부른다 ([LLM → 도구] 로그가 두 번 찍힌다)
print("\nQ: 2번 작업을 '긴급' 우선순위로 바꿔줘")
print("A:", await run_agent("2번 작업을 '긴급' 우선순위로 바꿔줘"))

## Lab 8 · Sampling — 서버가 클라이언트의 LLM으로 답한다
Lab 7은 **클라이언트가** LLM을 돌렸습니다. **Sampling**은 반대로 — **서버 도구가** 클라이언트의 LLM에게 "이 근거로 답해줘"라고 **요청**합니다. 서버는 자기 API 키·LLM 없이 클라이언트 것을 빌립니다.
- `ask(query)` 도구: 검색으로 근거를 모아 → `ctx.sample()`로 클라이언트 LLM(Qwen)에 답 요청 → 반환 (서버측 RAG)
- Client에 `sampling_handler`를 달아 그 요청을 Qwen으로 처리합니다.


**① ask 도구 — 검색 근거로 클라이언트 LLM에 답 요청**

In [ ]:
# ask 도구: 검색한 근거로 '클라이언트의 LLM'에게 답을 요청한다 (서버는 자기 LLM 없이 빌림)
@mcp.tool
async def ask(question: str, ctx: Context, k: int = 3) -> str:
    """검색 근거를 모아 클라이언트 LLM(sampling)에게 답을 요청한다 — 서버측 RAG"""
    hits = retrieve(question, k)                       # Lab 5의 벡터 검색 재사용
    evidence = "\n".join(f"- {h['text']}" for h in hits)
    resp = await ctx.sample(
        messages=f"다음 근거만 사용해 한국어로 간결히 답하라.\n[근거]\n{evidence}\n\n[질문] {question}",
        system_prompt="너는 사내 문서 도우미다. 근거에 없으면 '문서에 없음'이라 답한다.",
    )
    return resp.text

**② sampling 핸들러(Qwen) + 실행**
Client에 핸들러를 달면 ask의 `ctx.sample`이 Qwen으로 처리됩니다.

In [ ]:
# 클라이언트가 sampling 요청을 받으면 Qwen으로 처리하는 핸들러 (params는 camelCase)
async def qwen_sampler(messages, params, context):
    sys = params.systemPrompt or "You are a helpful assistant."
    user = "\n".join(m.content.text for m in messages if getattr(m.content, "text", None))
    r = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": sys}, {"role": "user", "content": user}],
        temperature=params.temperature or 0.2,
        max_tokens=params.maxTokens or 400)
    return r.choices[0].message.content

# Client에 sampling 핸들러를 달면 → 서버의 ask 도구가 이 LLM으로 답한다
async with Client(mcp, sampling_handler=qwen_sampler) as c:
    print("Q: 연차는 며칠 전에 신청해야 하나요?")
    print("A:", (await c.call_tool("ask", {"question": "연차는 며칠 전에 신청해야 하나요?"})).data)

## Lab 9 · 안전장치 — 되돌릴 수 없는 도구는 먼저 확인받는다
삭제처럼 위험한 도구엔 두 겹 안전장치를 답니다.
- `annotations={"destructiveHint": True}` — 이 도구가 **파괴적**임을 클라이언트에 알림
- `ctx.elicit()` — 실행 **전에 사용자에게 확인**을 받는다 (Elicitation)
> 실행하면 y/n 입력을 묻습니다 — n(또는 그 외)이면 삭제가 막힙니다.


**① 파괴적 도구 — destructiveHint + elicit로 실행 전 확인**

In [ ]:
from dataclasses import dataclass

FILES = {1: "old_report.pdf", 2: "temp_backup.zip"}

@dataclass
class Confirm:
    ok: bool   # 삭제를 진행할까요? (True/False)

@mcp.tool(annotations={"destructiveHint": True})   # 이 도구가 '파괴적'임을 클라이언트에 알림
async def delete_file(file_id: int, ctx: Context) -> str:
    """파일을 삭제한다 (되돌릴 수 없음) — 실행 전 사용자 확인을 받는다"""
    if file_id not in FILES:
        return f"파일 #{file_id}이 없습니다."
    res = await ctx.elicit(f"'{FILES[file_id]}'(#{file_id})을 정말 삭제할까요?", response_type=Confirm)
    if res.action != "accept" or not res.data.ok:
        return "삭제 취소됨 (안전장치가 막음)"
    name = FILES.pop(file_id)
    return f"삭제 완료: '{name}'"

**② 확인 핸들러(y/n) + 실행**
Client에 elicitation 핸들러를 달면 `delete_file`이 실행 전 사용자에게 y/n을 묻습니다.

In [ ]:
# 클라이언트가 확인 요청을 받으면 사용자에게 y/n 을 묻는다 (노트북 input)
async def confirm_handler(message, response_type, params, context):
    ans = input(f"[확인 요청] {message} (y/n): ")
    return {"ok": ans.strip().lower() in ("y", "yes", "예")}

# Client에 elicitation 핸들러를 달면 → 서버의 delete_file 이 실행 전 사용자에게 확인
async with Client(mcp, elicitation_handler=confirm_handler) as c:
    print(await c.call_tool("delete_file", {"file_id": 1}))   # y 입력 시 삭제, 그 외 취소
    print("남은 파일:", FILES)

## Lab 10 · 웹검색 — 에이전트가 외부 최신 정보까지
`web_search` 도구(키 없는 DuckDuckGo)를 붙이면, 에이전트가 **사내 지식(search_docs) + 웹 최신정보(web_search)** 를 질문에 따라 골라 씁니다.
- `openWorldHint=True` — 이 도구가 외부 세계에 접근함을 표시
> 아래 에이전트 데모는 Lab 7의 `run_agent` 를 재사용합니다(먼저 실행돼 있어야 함).


In [ ]:
%pip install -q ddgs

**① 웹검색 도구 (DuckDuckGo · 키 불필요)**

In [ ]:
from ddgs import DDGS

@mcp.tool(annotations={"readOnlyHint": True, "openWorldHint": True})   # 읽기 전용 + 외부 접근
def web_search(query: str, k: int = 3) -> list:
    """웹(DuckDuckGo)에서 최신 정보를 검색한다 — 키 불필요, 외부 접근"""
    with DDGS() as d:
        return [{"title": r["title"], "url": r["href"], "snippet": r.get("body", "")[:200]}
                for r in d.text(query, max_results=k)]

# 1) 도구 단독 확인
async with Client(mcp) as c:
    for r in (await c.call_tool("web_search", {"query": "Model Context Protocol 최신", "k": 3})).data:
        print("-", r["title"][:50], "|", r["url"][:50])

**② 멀티툴 에이전트 — 사내검색 + 웹검색을 골라 쓴다**

In [ ]:
# 2) 이제 에이전트(Lab 7의 run_agent)는 '사내검색(search_docs)'과 '웹검색(web_search)'을
#    질문에 따라 스스로 골라 쓴다 — 내부 지식 + 외부 최신 정보를 한 서버에서
print("\nQ: MCP의 최신 소식을 웹에서 찾아 한국어로 3줄 요약해줘")
print("A:", await run_agent("MCP의 최신 소식을 웹에서 찾아 한국어로 3줄 요약해줘"))

## 산출물 체크리스트
- [ ] 부작용 도구(저장/완료)와 읽기 전용 도구를 구분해 만들었다
- [ ] 잘못된 입력에 친절한 에러를 던진다
- [ ] 구조화된 출력(Pydantic)과 annotations(읽기 전용 등)를 붙였다
- [ ] Context로 진행률을 보고하고 클라이언트에서 받았다
- [ ] 검색을 Tool로 노출해 에이전트가 쓸 수 있게 했다
- [ ] LLM(Qwen)이 스스로 `search_docs`를 호출해 답하게 했다 (실무형 tool calling)
- [ ] 실제 코퍼스로 청킹·임베딩한 `search_docs`(RAG)와 `chunk://` 리소스를 만들었다
- [ ] `ask` 도구가 `ctx.sample()`로 클라이언트 LLM을 호출한다(sampling)
- [ ] 파괴적 도구에 `destructiveHint`를 달고 `ctx.elicit()`로 실행 전 확인을 받았다(안전장치)
- [ ] `web_search` 도구로 에이전트가 사내 지식 + 웹 최신정보를 함께 쓴다
